# 206 — TraceService
The single write surface for investigation trace data.
Agents call TraceService; nothing writes Cypher directly.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import uuid
import pandas as pd
from src.config import get_neo4j_settings
from src.storage.neo4j_repository import Neo4jRepository
from src.storage.trace_repository import TraceRepository
from src.tracing.trace_service import TraceService
from src.tools.trace_tools import TraceTools
from src.domain.models import (
    EventType,
    InvestigationRequest,
    UserContext,
)

settings = get_neo4j_settings()
neo4j    = Neo4jRepository(**vars(settings))
neo4j.test_connection()

repo    = TraceRepository(neo4j)
service = TraceService(repo)
tools   = TraceTools(repo)
print("Connected")

Connected


## Start a trace

In [3]:
request = InvestigationRequest(
    entity_name="TELEFONICA O2 HOLDINGS LIMITED",
    context=UserContext(
        user_id="analyst-001",
        session_id=str(uuid.uuid4()),
        metadata={"mode": "interactive"},
    ),
    request_id=str(uuid.uuid4()),
)

trace = service.start_trace(request, request.context)
print(f"trace_id : {trace.request_id}")
print(f"entity   : {trace.entity_name}")
print(f"started  : {trace.created_at}")

trace_id : e92618e6-04f2-4195-a305-be131296d5ee
entity   : TELEFONICA O2 HOLDINGS LIMITED
started  : 2026-03-22 02:52:31.548609+00:00


## Add events

In [4]:
# Agent reasoning — planning decision, no entity ref
service.create_agent_decision_event(
    trace,
    message="Plan created: run ownership check, then address check.",
    decision="start with ownership_complexity_check",
    why="ownership structure is the primary risk signal",
)

# Tool called — linked to the subject company
service.create_tool_event(
    trace,
    event_type=EventType.TOOL_CALLED,
    tool_name="ownership_complexity_check",
    step_id="step_1",
    input_summary="company_name=TELEFONICA O2 HOLDINGS LIMITED",
    entity_refs=[
        {"label": "Company", "name": "TELEFONICA O2 HOLDINGS LIMITED"},
    ],
)

# Tool returned — linked to subject + discovered owner
service.create_tool_event(
    trace,
    event_type=EventType.TOOL_RETURNED,
    tool_name="ownership_complexity_check",
    step_id="step_1",
    input_summary="company_name=TELEFONICA O2 HOLDINGS LIMITED",
    output_summary="risk_level=MEDIUM, max_depth=1, ubo_count=0",
    decision="proceed to address check",
    why="corporate chain only — no individual UBOs found",
    entity_refs=[
        {"label": "Company", "name": "TELEFONICA O2 HOLDINGS LIMITED"},
        {"label": "Company", "name": "Telefonica S A"},
    ],
)

# Step failed — tool raised an error
service.create_tool_event(
    trace,
    event_type=EventType.STEP_FAILED,
    tool_name="address_risk_check",
    step_id="step_2",
    message="address_risk_check failed: no address node found.",
    entity_refs=[
        {"label": "Company", "name": "TELEFONICA O2 HOLDINGS LIMITED"},
    ],
)

print(f"In-memory event count: {len(trace.events)}")

In-memory event count: 4


## Finalize

In [5]:
service.finalize_trace(
    trace,
    final_summary=(
        "Telefonica O2 Holdings Limited is wholly owned by Telefonica S.A. "
        "(75-100%). Address check failed. Ownership complexity: MEDIUM."
    ),
)
print(f"ended_at      : {trace.ended_at}")
print(f"final_summary : {trace.final_summary}")

ended_at      : 2026-03-22 02:53:13.050874+00:00
final_summary : Telefonica O2 Holdings Limited is wholly owned by Telefonica S.A. (75-100%). Address check failed. Ownership complexity: MEDIUM.


## Retrieve via TraceTools

In [6]:
r = tools.retrieve_trace(trace.request_id)
print(f"[{'✓' if r.success else '✗'}] {r.tool_name}  ({r.duration_ms} ms)")
print(f"    {r.summary}")
if r.data:
    print(f"\nfinal_summary: {r.data['final_summary']}")
    print('\nEvents:')
    events_df = pd.DataFrame(r.data["events"])
    display(events_df[["event_number", "event_type", "tool_name", "input_summary", "output_summary", "decision", "about"]])

[✓] retrieve_trace  (38.6 ms)
    Trace 'e92618e6-04f2-4195-a305-be131296d5ee' for 'TELEFONICA O2 HOLDINGS LIMITED': 4 event(s).

final_summary: Telefonica O2 Holdings Limited is wholly owned by Telefonica S.A. (75-100%). Address check failed. Ownership complexity: MEDIUM.

Events:


,event_number,event_type,tool_name,input_summary,output_summary,decision,about
0,1,agent_reasoning,,"Plan created: run ownership check, then addres...",,start with ownership_complexity_check,"[{'identifier': '', 'labels': None}]"
1,2,tool_called,ownership_complexity_check,company_name=TELEFONICA O2 HOLDINGS LIMITED,,,[{'identifier': 'TELEFONICA O2 HOLDINGS LIMITE...
2,3,tool_returned,ownership_complexity_check,company_name=TELEFONICA O2 HOLDINGS LIMITED,"risk_level=MEDIUM, max_depth=1, ubo_count=0",proceed to address check,[{'identifier': 'TELEFONICA O2 HOLDINGS LIMITE...
3,4,step_failed,address_risk_check,,,,[{'identifier': 'TELEFONICA O2 HOLDINGS LIMITE...


In [7]:
r = tools.find_traces_by_entity("TELEFONICA O2 HOLDINGS LIMITED")
print(f"[{'✓' if r.success else '✗'}] {r.tool_name}  ({r.duration_ms} ms)")
print(f"    {r.summary}")
if r.data:
    display(pd.DataFrame(r.data))

[✓] find_traces_by_entity  (41.6 ms)
    Found 1 trace(s) connected to 'TELEFONICA O2 HOLDINGS LIMITED'.


,trace_id,query,user_id,started_at,ended_at
0,e92618e6-04f2-4195-a305-be131296d5ee,TELEFONICA O2 HOLDINGS LIMITED,analyst-001,2026-03-22T02:52:31.548609+00:00,2026-03-22T02:53:13.050874+00:00


## Cleanup — delete this notebook's trace from Neo4j

In [8]:
# Delete the InvestigationTrace node, its TraceEvent nodes, and all relationships.
# Business nodes (Company, Address, etc.) are NOT deleted.
neo4j.run_query(
    """
    MATCH (t:InvestigationTrace {trace_id: $trace_id})
    OPTIONAL MATCH (t)-[:HAS_EVENT]->(e:TraceEvent)
    DETACH DELETE t, e
    """,
    {"trace_id": trace.request_id},
)
print(f"Deleted trace {trace.request_id} and its events")

Deleted trace e92618e6-04f2-4195-a305-be131296d5ee and its events


In [9]:
neo4j.close()
print("Driver closed")

Driver closed
